# STA-Net — n-back, 3-class, leave-one-session-out (seed 42)
Put this notebook next to `run_stanet_loso.py`, `stanet_model.py`, `stanet_common.py`, `crash_safe_logger.py` (and the other `stanet_*.py` files).

Scripts are launched through `run_script(...)` (sets `sys.argv` and runs the file as `__main__`), because `%run` does not expand `$variables`.

In [ ]:
import os, sys, runpy, torch

CODE_DIR  = r"E:\kntu\Thesis\codes\STA-Net"                               # folder with the .py files
DATA_ROOT = r"E:\kntu\Thesis\codes\Results\STANET_PAPER_EXACT_nback"     # Preproc_stanet_nback.py output
SEED = 42

os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

def run_script(name, *args):
    """Run a script in this kernel exactly like `python name args...` (argv set, __name__ == "__main__")."""
    old = sys.argv
    sys.argv = [name, *map(str, args)]
    try:
        runpy.run_path(name, run_name="__main__")
    finally:
        sys.argv = old

print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1) Smoke test — one subject (do this first)

In [ ]:
run_script("run_stanet_loso.py", "--data-root", DATA_ROOT, "--subjects", "1", "--seed", SEED)

## 2) Full run — all 26 subjects, seed 42
Results go to `<DATA_ROOT>\results_stanet_trial_seed42\` (CSV/JSON updated after every subject, checkpoints per fold). Extra flags, e.g. `"--val-mode", "cross_session"`, `"--adabn"`, `"--diagnostics"`, can be appended to the argument list.

In [ ]:
run_script("run_stanet_loso.py", "--data-root", DATA_ROOT, "--seed", SEED)

## 3) Trial-level evaluation (after step 2; add `"--adabn"` only if step 2 used it)

In [ ]:
run_script("stanet_trial_level_eval.py", "--data-root", DATA_ROOT, "--seed", SEED, "--val-mode", "trial")

## 4) Look at the summary

In [ ]:
import glob, pandas as pd
res = os.path.join(DATA_ROOT, f"results_stanet_trial_seed{SEED}")
csv = sorted(glob.glob(os.path.join(res, "stanet_all_subjects_*.csv")))[-1]
df = pd.read_csv(csv); display(df)
ok = df[df.status == "ok"]
print(f"{len(ok)} subjects | acc {ok.mean_acc.mean()*100:.2f} ± {ok.mean_acc.std(ddof=0)*100:.2f} % | kappa {ok.mean_kappa.mean():.3f}")